In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text processing
import re
import nltk

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)

# Warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
df = pd.read_excel('P719-Dataset.xlsx')

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()

In [ ]:
# Dataset information
df.info()

In [ ]:
# Statistical summary
df.describe(include='all')

In [ ]:
# Check missing values
df.isnull().sum()

In [ ]:
# Check duplicate records
df.duplicated().sum()

In [ ]:
df['rating'].value_counts().sort_index()

In [ ]:
# Create a working copy
df_clean = df.copy()

# Remove duplicate records
df_clean.drop_duplicates(inplace=True)

# Remove rows with missing essential information
df_clean.dropna(subset=['title', 'rating', 'body'], inplace=True)

# Remove leading and trailing spaces
df_clean['title'] = df_clean['title'].astype(str).str.strip()
df_clean['body'] = df_clean['body'].astype(str).str.strip()

# Validate ratings
df_clean = df_clean[df_clean['rating'].isin([1, 2, 3, 4, 5])]

print("Original Shape:", df.shape)
print("Cleaned Shape:", df_clean.shape)

In [ ]:
def assign_sentiment(rating):

    if rating <= 2:
        return 'Negative'

    elif rating == 3:
        return 'Neutral'

    else:
        return 'Positive'


df_clean['sentiment'] = df_clean['rating'].apply(assign_sentiment)

df_clean[['title', 'rating', 'body', 'sentiment']].head(10)

In [ ]:
df_clean['sentiment'].value_counts()

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df_clean,
    x='sentiment',
    order=['Negative', 'Neutral', 'Positive']
)

plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')

plt.show()

In [ ]:
# Check cleaned dataset dimensions
print("Dataset Shape:", df_clean.shape)

# Display column names
print("\nColumns:")
print(df_clean.columns.tolist())

# Display first five records
df_clean.head()

In [ ]:
# Count sentiment categories
sentiment_counts = df_clean['sentiment'].value_counts()

print(sentiment_counts)

# Calculate percentage
sentiment_percentage = (
    df_clean['sentiment'].value_counts(normalize=True) * 100
)

print("\nSentiment Percentage:")
print(sentiment_percentage.round(2))

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df_clean,
    x='sentiment',
    order=['Negative', 'Neutral', 'Positive']
)

plt.title('Distribution of Customer Sentiments')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')

plt.savefig('sentiment_distribution.png',
            dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df_clean,
    x='rating',
    order=[1, 2, 3, 4, 5]
)

plt.title('Customer Rating Distribution')
plt.xlabel('Rating (1-5 Stars)')
plt.ylabel('Number of Reviews')

plt.savefig('rating_distribution.png',
            dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Calculate review length
df_clean['review_length'] = df_clean['body'].astype(str).apply(len)

# Calculate word count
df_clean['word_count'] = df_clean['body'].astype(str).apply(
    lambda x: len(x.split())
)

# Statistical summary
df_clean[['review_length', 'word_count']].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df_clean,
    x='word_count',
    bins=30,
    kde=True
)

plt.title('Distribution of Review Word Counts')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')

plt.savefig('review_length_distribution.png',
            dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=df_clean,
    x='sentiment',
    y='word_count',
    order=['Negative', 'Neutral', 'Positive']
)

plt.title('Review Length by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Word Count')

plt.savefig('review_length_by_sentiment.png',
            dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Combine title and review body

df_clean['full_review'] = (
    df_clean['title'].astype(str) + ' ' +
    df_clean['body'].astype(str)
)

df_clean[['title', 'body', 'full_review']].head()

In [ ]:
import re

def clean_text(text):

    text = str(text).lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Keep Unicode letters and spaces
    text = re.sub(r'[^\w\s]', ' ', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


df_clean['clean_review'] = df_clean['full_review'].apply(clean_text)

df_clean[['full_review', 'clean_review']].head()

In [ ]:
from collections import Counter

all_words = ' '.join(df_clean['clean_review']).split()

word_counts = Counter(all_words)

# Display top 20 words
top_words = word_counts.most_common(20)

top_words

In [ ]:
words_df = pd.DataFrame(
    top_words,
    columns=['Word', 'Frequency']
)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=words_df,
    x='Frequency',
    y='Word'
)

plt.title('Top 20 Most Frequently Used Words')
plt.xlabel('Frequency')
plt.ylabel('Words')

plt.savefig('top_20_words.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

print("Total English stopwords:", len(stop_words))

In [ ]:
def remove_stopwords(text):

    words = text.split()

    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    return ' '.join(filtered_words)


df_clean['filtered_review'] = (
    df_clean['clean_review'].apply(remove_stopwords)
)

df_clean[['clean_review', 'filtered_review']].head()

In [ ]:
filtered_words = ' '.join(
    df_clean['filtered_review']
).split()

filtered_counts = Counter(filtered_words)

top_filtered_words = filtered_counts.most_common(20)

top_filtered_words

In [ ]:
filtered_df = pd.DataFrame(
    top_filtered_words,
    columns=['Word', 'Frequency']
)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=filtered_df,
    x='Frequency',
    y='Word'
)

plt.title('Top 20 Meaningful Words in Customer Reviews')
plt.xlabel('Frequency')
plt.ylabel('Words')

plt.savefig('top_20_meaningful_words.png',
            dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
for sentiment in ['Positive', 'Negative', 'Neutral']:

    sentiment_text = ' '.join(
        df_clean.loc[
            df_clean['sentiment'] == sentiment,
            'filtered_review'
        ]
    )

    words = sentiment_text.split()

    word_frequency = Counter(words)

    print(f"\nTop 15 Words in {sentiment} Reviews:")

    print(word_frequency.most_common(15))

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud

all_text = ' '.join(df_clean['filtered_review'])

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color='white',
    max_words=100
).generate(all_text)

plt.figure(figsize=(15, 7))

plt.imshow(wordcloud, interpolation='bilinear')

plt.axis('off')

plt.title('Overall Customer Review Word Cloud')

plt.savefig('overall_wordcloud.png',
            dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
for sentiment in ['Positive', 'Negative']:

    sentiment_text = ' '.join(
        df_clean.loc[
            df_clean['sentiment'] == sentiment,
            'filtered_review'
        ]
    )

    wordcloud = WordCloud(
        width=1200,
        height=600,
        background_color='white',
        max_words=100
    ).generate(sentiment_text)

    plt.figure(figsize=(15, 7))

    plt.imshow(wordcloud, interpolation='bilinear')

    plt.axis('off')

    plt.title(f'{sentiment} Review Word Cloud')

    plt.savefig(
        f'{sentiment.lower()}_wordcloud.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()

In [ ]:
sentiment_rating = pd.crosstab(
    df_clean['rating'],
    df_clean['sentiment']
)

sentiment_rating

In [ ]:
plt.figure(figsize=(9, 5))

sns.heatmap(
    sentiment_rating,
    annot=True,
    fmt='d',
    cmap='YlGnBu'
)

plt.title('Sentiment Distribution Across Customer Ratings')
plt.xlabel('Sentiment')
plt.ylabel('Customer Rating')

plt.savefig(
    'sentiment_rating_heatmap.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
review_length_analysis = df_clean.groupby('sentiment').agg(
    Average_Word_Count=('word_count', 'mean'),
    Maximum_Word_Count=('word_count', 'max'),
    Minimum_Word_Count=('word_count', 'min'),
    Total_Reviews=('word_count', 'count')
)

review_length_analysis.round(2)

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=df_clean,
    x='sentiment',
    y='word_count',
    order=['Negative', 'Neutral', 'Positive'],
    estimator='mean'
)

plt.title('Average Review Length by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Average Number of Words')

plt.savefig(
    'average_review_length.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
for sentiment in ['Positive', 'Negative', 'Neutral']:

    print(f"\n{'='*60}")
    print(f"{sentiment.upper()} REVIEWS")
    print('='*60)

    samples = df_clean[
        df_clean['sentiment'] == sentiment
    ][['title', 'rating', 'body']].sample(
        min(3, len(df_clean[df_clean['sentiment'] == sentiment])),
        random_state=42
    )

    for index, row in samples.iterrows():

        print("\nRating:", row['rating'])
        print("Title:", row['title'])
        print("Review:", row['body'])
        print("-"*60)

In [ ]:
class_distribution = df_clean['sentiment'].value_counts()

class_percentage = (
    df_clean['sentiment'].value_counts(normalize=True) * 100
)

class_summary = pd.DataFrame({
    'Review_Count': class_distribution,
    'Percentage': class_percentage
})

class_summary.round(2)

In [ ]:
df_clean.to_csv(
    'P719_Cleaned_EDA_Dataset.csv',
    index=False
)

print("Cleaned dataset exported successfully!")

In [ ]:
from google.colab import files

files.download('P719_Cleaned_EDA_Dataset.csv')

In [ ]:
# Create a separate modeling dataset

df_model = df_clean.copy()

# Combine title and body
df_model['review_text'] = (
    df_model['title'].astype(str) + ' ' +
    df_model['body'].astype(str)
)

# Keep only required columns
df_model = df_model[
    ['review_text', 'rating', 'sentiment']
].copy()

# Check dataset
print("Dataset Shape:", df_model.shape)

df_model.head()

In [ ]:
# Check empty reviews

print(
    "Empty Reviews:",
    df_model['review_text'].str.strip().eq('').sum()
)

# Check missing values

print("\nMissing Values:")
print(df_model.isnull().sum())

In [ ]:
df_model['character_count'] = (
    df_model['review_text'].str.len()
)

df_model['word_count'] = (
    df_model['review_text'].str.split().str.len()
)

df_model[['character_count', 'word_count']].describe()

In [ ]:
import re

def preprocess_text(text):

    # Convert to lowercase
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


df_model['processed_text'] = (
    df_model['review_text'].apply(preprocess_text)
)

df_model[['review_text', 'processed_text']].head()

In [ ]:
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):

    words = text.split()

    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    return ' '.join(filtered_words)


df_model['final_text'] = (
    df_model['processed_text'].apply(remove_stopwords)
)

df_model[['processed_text', 'final_text']].head()

In [ ]:
negation_words = {'no', 'nor', 'not', 'never'}

stop_words = set(stopwords.words('english')) - negation_words

df_model['final_text'] = (
    df_model['processed_text'].apply(remove_stopwords)
)

In [ ]:
for i in range(5):

    print("Original Review:")
    print(df_model['review_text'].iloc[i])

    print("\nProcessed Review:")
    print(df_model['final_text'].iloc[i])

    print("\nSentiment:")
    print(df_model['sentiment'].iloc[i])

    print("-" * 80)

In [ ]:
# Define input features and target variable

X = df_model['final_text']
y = df_model['sentiment']

print("Input Features Shape:", X.shape)
print("Target Variable Shape:", y.shape)

print("\nTarget Distribution:")
print(y.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

print("\nTraining Sentiment Distribution:")
print(y_train.value_counts())

print("\nTesting Sentiment Distribution:")
print(y_test.value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Fit only on training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform testing data
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF Vectorization Completed!")

print("\nTraining Matrix Shape:", X_train_tfidf.shape)
print("Testing Matrix Shape:", X_test_tfidf.shape)

print("\nVocabulary Size:", len(tfidf.vocabulary_))

In [ ]:
feature_names = tfidf.get_feature_names_out()

print("Total Features:", len(feature_names))

print("\nFirst 30 Features:")
print(feature_names[:30])

In [ ]:
tfidf_sample = pd.DataFrame(
    X_train_tfidf[:5].toarray(),
    columns=feature_names
)

tfidf_sample.iloc[:, :20]

In [ ]:
vocabulary_frequency = (
    X_train_tfidf.getnnz(axis=0)
)

vocabulary_df = pd.DataFrame({
    'Word': feature_names,
    'Document_Frequency': vocabulary_frequency
})

vocabulary_df = vocabulary_df.sort_values(
    by='Document_Frequency',
    ascending=False
)

vocabulary_df.head(20)

In [ ]:
top_vocabulary = vocabulary_df.head(20)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=top_vocabulary,
    x='Document_Frequency',
    y='Word'
)

plt.title('Top 20 TF-IDF Vocabulary Terms')
plt.xlabel('Number of Documents')
plt.ylabel('Words and Phrases')

plt.savefig(
    'tfidf_top_vocabulary.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
print("FINAL DATASET SUMMARY")
print("=" * 45)

print("Original Dataset:", df_model.shape)

print("\nTraining Data:")
print("X_train:", X_train_tfidf.shape)
print("y_train:", y_train.shape)

print("\nTesting Data:")
print("X_test:", X_test_tfidf.shape)
print("y_test:", y_test.shape)

print("\nFeature Count:", X_train_tfidf.shape[1])

In [ ]:
import joblib

# Save TF-IDF vectorizer
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

# Save training and testing matrices
joblib.dump(X_train_tfidf, 'X_train_tfidf.pkl')
joblib.dump(X_test_tfidf, 'X_test_tfidf.pkl')

# Save target variables
joblib.dump(y_train, 'y_train.pkl')
joblib.dump(y_test, 'y_test.pkl')

print("Feature engineering artifacts saved successfully!")

In [ ]:
# Machine Learning Models

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Evaluation Metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Machine learning libraries imported successfully!")

In [ ]:
models = {

    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ),

    'Multinomial Naive Bayes': MultinomialNB(
        alpha=1.0
    ),

    'Linear SVM': LinearSVC(
        class_weight='balanced',
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
}

print("Four machine learning models initialized!")

In [ ]:
trained_models = {}

for name, model in models.items():

    print(f"\nTraining: {name}")

    model.fit(X_train_tfidf, y_train)

    trained_models[name] = model

    print(f"{name} trained successfully!")

print("\nAll models training completed!")

In [ ]:
predictions = {}

for name, model in trained_models.items():

    y_pred = model.predict(X_test_tfidf)

    predictions[name] = y_pred

    print(f"{name}: Predictions completed")

In [ ]:
results = []

for name, y_pred in predictions.items():

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average='weighted',
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average='weighted',
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average='weighted',
        zero_division=0
    )

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='F1-Score',
    ascending=False
).reset_index(drop=True)

results_df.round(4)

In [ ]:
metrics_df = results_df.set_index('Model')

metrics_df.plot(
    kind='bar',
    figsize=(12, 6)
)

plt.title('Machine Learning Model Performance Comparison')

plt.xlabel('Machine Learning Models')
plt.ylabel('Performance Score')

plt.xticks(rotation=20)

plt.legend(loc='lower right')

plt.tight_layout()

plt.savefig(
    'model_performance_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
for name, y_pred in predictions.items():

    print("\n" + "=" * 60)

    print(f"CLASSIFICATION REPORT: {name}")

    print("=" * 60)

    print(
        classification_report(
            y_test,
            y_pred,
            labels=['Negative', 'Neutral', 'Positive'],
            zero_division=0
        )
    )

In [ ]:
for name, y_pred in predictions.items():

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=['Negative', 'Neutral', 'Positive']
    )

    plt.figure(figsize=(7, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Negative', 'Neutral', 'Positive'],
        yticklabels=['Negative', 'Neutral', 'Positive']
    )

    plt.title(f'Confusion Matrix - {name}')

    plt.xlabel('Predicted Sentiment')

    plt.ylabel('Actual Sentiment')

    plt.savefig(
        f'confusion_matrix_{name.lower().replace(" ", "_")}.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()

In [ ]:
best_model_name = results_df.iloc[0]['Model']

best_model = trained_models[best_model_name]

print("Best Performing Model:", best_model_name)

print(
    "Weighted F1-Score:",
    round(results_df.iloc[0]['F1-Score'], 4)
)

print(
    "Accuracy:",
    round(results_df.iloc[0]['Accuracy'], 4)
)

In [ ]:
best_model_name = results_df.iloc[0]['Model']

best_model = trained_models[best_model_name]

print("Best Performing Model:", best_model_name)

print(
    "Weighted F1-Score:",
    round(results_df.iloc[0]['F1-Score'], 4)
)

print(
    "Accuracy:",
    round(results_df.iloc[0]['Accuracy'], 4)
)

In [ ]:
import joblib
import os

os.makedirs('saved_models', exist_ok=True)

# Save all trained models

for name, model in trained_models.items():

    filename = name.lower().replace(' ', '_') + '.pkl'

    joblib.dump(
        model,
        f'saved_models/{filename}'
    )

# Save best model

joblib.dump(
    best_model,
    'saved_models/best_sentiment_model.pkl'
)

# Save TF-IDF vectorizer

joblib.dump(
    tfidf,
    'saved_models/tfidf_vectorizer.pkl'
)

print("All models and vectorizer saved successfully!")

In [ ]:
from sklearn.metrics import f1_score

macro_results = []

for name, y_pred in predictions.items():

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average='macro'
    )

    weighted_f1 = f1_score(
        y_test,
        y_pred,
        average='weighted'
    )

    macro_results.append({
        'Model': name,
        'Macro_F1': macro_f1,
        'Weighted_F1': weighted_f1
    })

macro_results_df = pd.DataFrame(macro_results)

macro_results_df.sort_values(
    by='Macro_F1',
    ascending=False
).round(4)

In [ ]:
from sklearn.metrics import classification_report

best_model_name = macro_results_df.sort_values(
    'Macro_F1',
    ascending=False
).iloc[0]['Model']

best_model = trained_models[best_model_name]

best_predictions = predictions[best_model_name]

print("Selected Model:", best_model_name)

print(
    classification_report(
        y_test,
        best_predictions,
        labels=['Negative', 'Neutral', 'Positive'],
        zero_division=0
    )
)

In [ ]:
test_reviews = [
    "This phone is absolutely amazing. Great camera and battery life.",
    "Very disappointed with this product. Poor performance and terrible battery.",
    "The phone is okay. Nothing special about the performance.",
    "Excellent quality and value for money.",
    "Worst purchase ever. Completely dissatisfied."
]

# Apply the same preprocessing
processed_reviews = [
    remove_stopwords(preprocess_text(review))
    for review in test_reviews
]

# Convert text into TF-IDF features
review_vectors = tfidf.transform(processed_reviews)

# Predict sentiment
new_predictions = best_model.predict(review_vectors)

# Display results
prediction_results = pd.DataFrame({
    'Customer_Review': test_reviews,
    'Predicted_Sentiment': new_predictions
})

prediction_results

In [ ]:
import joblib
import os

os.makedirs('final_model', exist_ok=True)

joblib.dump(
    best_model,
    'final_model/sentiment_model.pkl'
)

joblib.dump(
    tfidf,
    'final_model/tfidf_vectorizer.pkl'
)

joblib.dump(
    {
        'preprocessing': 'preprocess_text + remove_stopwords',
        'sentiment_classes': ['Negative', 'Neutral', 'Positive'],
        'selected_model': best_model_name
    },
    'final_model/model_config.pkl'
)

print("Final model artifacts saved successfully!")

In [ ]:
import os
import joblib

# Create deployment folder
os.makedirs('deployment', exist_ok=True)

# Save best model
joblib.dump(
    best_model,
    'deployment/sentiment_model.pkl'
)

# Save TF-IDF vectorizer
joblib.dump(
    tfidf,
    'deployment/tfidf_vectorizer.pkl'
)

# Save model information
model_info = {
    'model_name': best_model_name,
    'classes': ['Negative', 'Neutral', 'Positive']
}

joblib.dump(
    model_info,
    'deployment/model_info.pkl'
)

print("Deployment files saved successfully!")

In [ ]:
import os

print("Deployment Files:")

for file in os.listdir('deployment'):
    print(file)

In [ ]:
import shutil

shutil.make_archive(
    'P719_Sentiment_Model',
    'zip',
    'deployment'
)

from google.colab import files

files.download('P719_Sentiment_Model.zip')

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import re
import matplotlib.pyplot as plt

# -----------------------------------
# 1. PAGE CONFIGURATION
# -----------------------------------

st.set_page_config(
    page_title="Customer Sentiment Analysis",
    page_icon="📊",
    layout="wide"
)

# -----------------------------------
# 2. LOAD TRAINED MODEL
# -----------------------------------

@st.cache_resource
def load_model():

    model = joblib.load("sentiment_model.pkl")
    tfidf = joblib.load("tfidf_vectorizer.pkl")

    return model, tfidf


model, tfidf = load_model()

# -----------------------------------
# 3. TEXT PREPROCESSING
# -----------------------------------

def preprocess_text(text):

    text = str(text).lower()

    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Preserve the same stopword configuration
import nltk
from nltk.corpus import stopwords

try:
    stop_words = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))

negation_words = {'no', 'nor', 'not', 'never'}
stop_words = stop_words - negation_words


def remove_stopwords(text):

    return ' '.join(
        word for word in text.split()
        if word not in stop_words
    )


def prepare_text(text):

    cleaned = preprocess_text(text)

    return remove_stopwords(cleaned)


# -----------------------------------
# 4. APPLICATION HEADER
# -----------------------------------

st.title("📊 Customer Review Sentiment Analysis")

st.write(
    "Analyze customer reviews using Natural Language Processing "
    "and Machine Learning."
)

st.divider()

# -----------------------------------
# 5. NAVIGATION
# -----------------------------------

menu = st.sidebar.selectbox(
    "Navigation",
    [
        "Home",
        "Single Review Prediction",
        "Bulk Review Analysis",
        "Sentiment Dashboard"
    ]
)

# -----------------------------------
# 6. HOME PAGE
# -----------------------------------

if menu == "Home":

    st.header("Welcome")

    st.write(
        "This application predicts whether customer reviews "
        "express Positive, Negative or Neutral sentiment."
    )

    col1, col2, col3 = st.columns(3)

    col1.metric("Analysis Type", "NLP")
    col2.metric("Classification", "3 Classes")
    col3.metric("ML Model", type(model).__name__)

    st.info(
        "Select an option from the sidebar to begin."
    )


# -----------------------------------
# 7. SINGLE REVIEW PREDICTION
# -----------------------------------

elif menu == "Single Review Prediction":

    st.header("Single Review Sentiment Prediction")

    review = st.text_area(
        "Enter Customer Review",
        placeholder="Write your product review here..."
    )

    if st.button("Analyze Sentiment"):

        if review.strip():

            processed = prepare_text(review)

            vector = tfidf.transform([processed])

            prediction = model.predict(vector)[0]

            st.subheader("Prediction Result")

            if prediction == "Positive":
                st.success("Positive Sentiment 😊")

            elif prediction == "Negative":
                st.error("Negative Sentiment 😞")

            else:
                st.warning("Neutral Sentiment 😐")

        else:

            st.warning("Please enter a customer review.")


# -----------------------------------
# 8. BULK REVIEW ANALYSIS
# -----------------------------------

elif menu == "Bulk Review Analysis":

    st.header("Bulk Customer Review Analysis")

    uploaded_file = st.file_uploader(
        "Upload CSV File",
        type=["csv"]
    )

    if uploaded_file is not None:

        data = pd.read_csv(uploaded_file)

        st.write("Uploaded Dataset")
        st.dataframe(data.head())

        review_column = st.selectbox(
            "Select Review Column",
            data.columns
        )

        if st.button("Analyze All Reviews"):

            valid_reviews = data[review_column].fillna('').astype(str)

            processed = valid_reviews.apply(prepare_text)

            vectors = tfidf.transform(processed)

            data["Predicted_Sentiment"] = model.predict(vectors)

            st.success("Sentiment analysis completed!")

            st.dataframe(data)

            csv = data.to_csv(index=False).encode("utf-8")

            st.download_button(
                "Download Results",
                data=csv,
                file_name="sentiment_predictions.csv",
                mime="text/csv"
            )


# -----------------------------------
# 9. SENTIMENT DASHBOARD
# -----------------------------------

elif menu == "Sentiment Dashboard":

    st.header("Sentiment Analytics Dashboard")

    uploaded_file = st.file_uploader(
        "Upload CSV File",
        type=["csv"],
        key="dashboard_upload"
    )

    if uploaded_file is not None:

        data = pd.read_csv(uploaded_file)

        review_column = st.selectbox(
            "Select Review Column",
            data.columns,
            key="dashboard_column"
        )

        if st.button("Generate Dashboard"):

            valid_reviews = data[review_column].fillna('').astype(str)

            processed = valid_reviews.apply(prepare_text)

            vectors = tfidf.transform(processed)

            data["Predicted_Sentiment"] = model.predict(vectors)

            counts = data["Predicted_Sentiment"].value_counts()

            col1, col2, col3 = st.columns(3)

            col1.metric(
                "Positive Reviews",
                int(counts.get("Positive", 0))
            )

            col2.metric(
                "Negative Reviews",
                int(counts.get("Negative", 0))
            )

            col3.metric(
                "Neutral Reviews",
                int(counts.get("Neutral", 0))
            )

            st.subheader("Sentiment Distribution")

            st.bar_chart(counts)

            st.subheader("Prediction Results")

            st.dataframe(data)

            csv = data.to_csv(index=False).encode("utf-8")

            st.download_button(
                "Download Dashboard Results",
                data=csv,
                file_name="sentiment_dashboard_results.csv",
                mime="text/csv"
            )

st.divider()

st.caption(
    "NLP Sentiment Analysis Project | Customer Review Classification"
)

In [ ]:
%%writefile requirements.txt

streamlit
pandas
numpy
scikit-learn
joblib
matplotlib
nltk
openpyxl

In [ ]:
import os

print("Project Files:")

for file in os.listdir():
    if file in ['app.py', 'requirements.txt']:
        print("✔", file)

In [ ]:
import os
import shutil

project_dir = "P719-NLP-Sentiment-Analysis"

os.makedirs(project_dir, exist_ok=True)
os.makedirs(f"{project_dir}/notebooks", exist_ok=True)
os.makedirs(f"{project_dir}/visualizations", exist_ok=True)

# Copy application files
shutil.copy("app.py", project_dir)
shutil.copy("requirements.txt", project_dir)

# Copy model files from deployment folder
shutil.copy(
    "deployment/sentiment_model.pkl",
    project_dir
)

shutil.copy(
    "deployment/tfidf_vectorizer.pkl",
    project_dir
)

print("Project structure created!")

In [ ]:
import glob

for file in glob.glob("*.png"):
    shutil.copy(file, f"{project_dir}/visualizations")

print("Visualizations copied!")